# Marketing Campaign Analysis

This notebook analyzes `Marketing_Campaign_Data.csv` to answer the client question:

**Which campaign and channel combination should we focus on to increase sales to new customers, and why?**

- Campaign A: informal and conversational tone
- Campaign B: formal and corporative tone

The workflow below loads the dataset, compares new-customer performance by campaign and channel, builds visualizations, and exports an HTML dashboard.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_PATH = Path(r"c:\Users\davie\OneDrive\Documentos\Data Analyst\Marketing_Campaign_Data.csv")
OUTPUT_HTML = Path(r"c:\Users\davie\OneDrive\Documentos\Data Analyst\marketing_campaign_dashboard_from_notebook.html")

# Load and standardize the source data.
df = pd.read_csv(DATA_PATH)
df.columns = [column.strip() for column in df.columns]

numeric_columns = [
    "Converted (1=yes, 0=no)",
    "Time on Site (seconds)",
    "Sales ($)",
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

new_customers = df[df["Customer Type"].eq("New")].copy()
new_customers.head()

,Interaction ID,Campaign Type,Channel,Customer Type,"Converted (1=yes, 0=no)",Time on Site (seconds),Sales ($)
1,INT000002,B,Email,New,1,168.161435,61.085973
2,INT000003,A,Email,New,0,146.636771,0.000000
12,INT000013,B,Email,New,1,215.246637,43.438914
13,INT000014,B,Instagram,New,1,145.739910,34.841629
15,INT000016,B,Instagram,New,0,117.937220,0.000000


In [2]:
overall_kpis = {
    "new_customer_rows": int(len(new_customers)),
    "new_customer_sales": float(new_customers["Sales ($)"].sum()),
    "new_customer_conversion_rate": float(new_customers["Converted (1=yes, 0=no)"].mean()),
}

combo_summary = (
    new_customers.groupby(["Campaign Type", "Channel"], as_index=False)
    .agg(
        leads=("Interaction ID", "count"),
        conversions=("Converted (1=yes, 0=no)", "sum"),
        conversion_rate=("Converted (1=yes, 0=no)", "mean"),
        total_sales=("Sales ($)", "sum"),
        sales_per_lead=("Sales ($)", "mean"),
        avg_time_on_site=("Time on Site (seconds)", "mean"),
    )
)
combo_summary["sales_share"] = combo_summary["total_sales"] / combo_summary["total_sales"].sum()
combo_summary["focus_score"] = (
    0.55 * (combo_summary["total_sales"] / combo_summary["total_sales"].mean())
    + 0.45 * (combo_summary["sales_per_lead"] / combo_summary["sales_per_lead"].mean())
)
combo_summary = combo_summary.sort_values(
    ["focus_score", "total_sales", "conversion_rate"],
    ascending=[False, False, False],
).reset_index(drop=True)

campaign_summary = (
    new_customers.groupby("Campaign Type", as_index=False)
    .agg(
        leads=("Interaction ID", "count"),
        conversions=("Converted (1=yes, 0=no)", "sum"),
        conversion_rate=("Converted (1=yes, 0=no)", "mean"),
        total_sales=("Sales ($)", "sum"),
        sales_per_lead=("Sales ($)", "mean"),
    )
    .sort_values("total_sales", ascending=False)
)

channel_summary = (
    new_customers.groupby("Channel", as_index=False)
    .agg(
        leads=("Interaction ID", "count"),
        conversions=("Converted (1=yes, 0=no)", "sum"),
        conversion_rate=("Converted (1=yes, 0=no)", "mean"),
        total_sales=("Sales ($)", "sum"),
        sales_per_lead=("Sales ($)", "mean"),
    )
    .sort_values("total_sales", ascending=False)
)

recommended_combo = combo_summary.iloc[0]
recommended_combo.to_frame(name="value")

,value
Campaign Type,B
Channel,Email
leads,9700
conversions,4673
conversion_rate,0.481753
total_sales,225285.520362
sales_per_lead,23.225311
avg_time_on_site,134.545097
sales_share,0.2279
focus_score,1.202454


## Recommendation

**Focus on Campaign B through Email.**

Why this combination wins for new customers:

- It generates the highest total sales in the dataset.
- It converts above the overall new-customer average.
- It combines strong efficiency with enough lead volume to scale.
- Campaign B, the more formal and corporative tone, outperforms Campaign A on total new-customer sales across all channels.

In [3]:
display_columns = [
    "Campaign Type",
    "Channel",
    "leads",
    "conversions",
    "conversion_rate",
    "total_sales",
    "sales_per_lead",
    "sales_share",
]

styled_combo_summary = combo_summary[display_columns].copy()
styled_combo_summary["conversion_rate"] = styled_combo_summary["conversion_rate"].map(lambda value: f"{value:.2%}")
styled_combo_summary["total_sales"] = styled_combo_summary["total_sales"].map(lambda value: f"${value:,.2f}")
styled_combo_summary["sales_per_lead"] = styled_combo_summary["sales_per_lead"].map(lambda value: f"${value:,.2f}")
styled_combo_summary["sales_share"] = styled_combo_summary["sales_share"].map(lambda value: f"{value:.2%}")

print("Overall new-customer KPIs")
print(f"Rows: {overall_kpis['new_customer_rows']:,}")
print(f"Sales: ${overall_kpis['new_customer_sales']:,.2f}")
print(f"Conversion rate: {overall_kpis['new_customer_conversion_rate']:.2%}")
print()
print("Ranked campaign-channel combinations")
styled_combo_summary

Overall new-customer KPIs
Rows: 42,597
Sales: $988,528.51
Conversion rate: 47.86%

Ranked campaign-channel combinations


,Campaign Type,Channel,leads,conversions,conversion_rate,total_sales,sales_per_lead,sales_share
0,B,Email,9700,4673,48.18%,"$225,285.52",$23.23,22.79%
1,A,Email,7683,3628,47.22%,"$177,366.97",$23.09,17.94%
2,B,Instagram,7100,3429,48.30%,"$166,722.17",$23.48,16.87%
3,B,Web Banner,7110,3387,47.64%,"$164,125.79",$23.08,16.60%
4,A,Instagram,5456,2658,48.72%,"$127,814.93",$23.43,12.93%
5,A,Web Banner,5548,2614,47.12%,"$127,213.12",$22.93,12.87%


In [4]:
color_map = {"A": "#c96b59", "B": "#2f7f7b"}

top_order = combo_summary[["Campaign Type", "Channel"]].copy()
top_order["label"] = top_order["Campaign Type"] + " - " + top_order["Channel"]
combo_chart_data = combo_summary.copy()
combo_chart_data["label"] = combo_chart_data["Campaign Type"] + " - " + combo_chart_data["Channel"]
combo_chart_data["tone"] = combo_chart_data["Campaign Type"].map({
    "A": "Informal / conversational",
    "B": "Formal / corporative",
})

bar_fig = px.bar(
    combo_chart_data,
    x="label",
    y="total_sales",
    color="Campaign Type",
    text="total_sales",
    color_discrete_map=color_map,
    category_orders={"label": top_order["label"].tolist()},
    hover_data={
        "Channel": True,
        "tone": True,
        "leads": ':,',
        "conversion_rate": ':.2%',
        "sales_per_lead": ':$.2f',
        "total_sales": ':$.2f',
        "label": False,
    },
    title="New-customer sales by campaign and channel",
)
bar_fig.update_traces(texttemplate="$%{y:,.0f}", textposition="outside")
bar_fig.update_layout(
    xaxis_title="Campaign and channel",
    yaxis_title="Sales ($)",
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend_title_text="Campaign",
)
bar_fig.show()

scatter_fig = px.scatter(
    combo_chart_data,
    x="conversion_rate",
    y="sales_per_lead",
    size="leads",
    color="Campaign Type",
    color_discrete_map=color_map,
    hover_name="label",
    hover_data={
        "tone": True,
        "total_sales": ':$.2f',
        "conversion_rate": ':.2%',
        "sales_per_lead": ':$.2f',
        "leads": ':,',
    },
    title="Scale versus efficiency for new-customer combinations",
)
scatter_fig.update_layout(
    xaxis_title="Conversion rate",
    yaxis_title="Sales per lead ($)",
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend_title_text="Campaign",
)
scatter_fig.show()

In [5]:
comparison = (
    combo_summary.pivot(index="Channel", columns="Campaign Type", values=["total_sales", "conversion_rate"])
    .sort_index()
)
comparison_table = pd.DataFrame({
    "Channel": comparison.index,
    "B_minus_A_sales": comparison[("total_sales", "B")] - comparison[("total_sales", "A")],
    "B_minus_A_conversion_pp": (comparison[("conversion_rate", "B")] - comparison[("conversion_rate", "A")]) * 100,
}).reset_index(drop=True)
comparison_table

comparison_fig = make_subplots(specs=[[{"secondary_y": True}]])
comparison_fig.add_trace(
    go.Bar(
        x=comparison_table["Channel"],
        y=comparison_table["B_minus_A_sales"],
        name="Revenue lift from B vs A",
        marker_color="#20364f",
        hovertemplate="%{x}<br>Revenue lift: $%{y:,.2f}<extra></extra>",
    ),
    secondary_y=False,
)
comparison_fig.add_trace(
    go.Scatter(
        x=comparison_table["Channel"],
        y=comparison_table["B_minus_A_conversion_pp"],
        name="Conversion delta (pp)",
        mode="lines+markers+text",
        text=[f"{value:+.2f} pp" for value in comparison_table["B_minus_A_conversion_pp"]],
        textposition="top center",
        line=dict(color="#ca8f2c", width=3),
        marker=dict(size=9),
        hovertemplate="%{x}<br>Conversion delta: %{y:.2f} pp<extra></extra>",
    ),
    secondary_y=True,
)
comparison_fig.update_layout(
    title="Campaign B uplift versus Campaign A by channel",
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
comparison_fig.update_yaxes(title_text="Revenue lift ($)", secondary_y=False)
comparison_fig.update_yaxes(title_text="Conversion delta (pp)", secondary_y=True)
comparison_fig.show()

In [6]:
dashboard_fig = make_subplots(
    rows=2,
    cols=2,
    specs=[[{"type": "bar"}, {"type": "scatter"}], [{"type": "bar"}, {"type": "table"}]],
    subplot_titles=(
        "Sales by campaign and channel",
        "Scale versus efficiency",
        "Campaign B uplift versus A",
        "Ranked combinations",
    ),
    vertical_spacing=0.14,
)

for campaign in ["A", "B"]:
    campaign_slice = combo_chart_data[combo_chart_data["Campaign Type"] == campaign]
    dashboard_fig.add_trace(
        go.Bar(
            x=campaign_slice["label"],
            y=campaign_slice["total_sales"],
            name=f"Campaign {campaign}",
            marker_color=color_map[campaign],
            showlegend=True,
        ),
        row=1,
        col=1,
    )

dashboard_fig.add_trace(
    go.Scatter(
        x=combo_chart_data["conversion_rate"],
        y=combo_chart_data["sales_per_lead"],
        mode="markers+text",
        text=combo_chart_data["label"],
        textposition="top center",
        marker=dict(
            size=(combo_chart_data["leads"] / 180).round(1),
            color=combo_chart_data["Campaign Type"].map(color_map),
            sizemode="diameter",
            line=dict(color="white", width=1),
        ),
        name="Combinations",
        showlegend=False,
    ),
    row=1,
    col=2,
)

dashboard_fig.add_trace(
    go.Bar(
        x=comparison_table["Channel"],
        y=comparison_table["B_minus_A_sales"],
        marker_color="#20364f",
        name="Revenue lift",
        showlegend=False,
    ),
    row=2,
    col=1,
)

dashboard_fig.add_trace(
    go.Table(
        header=dict(
            values=["Campaign", "Channel", "Leads", "Conv. Rate", "Sales"],
            fill_color="#20364f",
            font=dict(color="white", size=12),
            align="left",
        ),
        cells=dict(
            values=[
                combo_summary["Campaign Type"],
                combo_summary["Channel"],
                combo_summary["leads"].map(lambda value: f"{value:,}"),
                combo_summary["conversion_rate"].map(lambda value: f"{value:.2%}"),
                combo_summary["total_sales"].map(lambda value: f"${value:,.2f}"),
            ],
            fill_color="#f8f5ef",
            align="left",
            height=28,
        ),
    ),
    row=2,
    col=2,
)

dashboard_fig.update_layout(
    height=980,
    width=1400,
    title_text="Marketing Campaign Dashboard: New-Customer Acquisition",
    plot_bgcolor="white",
    paper_bgcolor="white",
)

dashboard_fig.update_xaxes(title_text="Campaign and channel", row=1, col=1)
dashboard_fig.update_yaxes(title_text="Sales ($)", row=1, col=1)
dashboard_fig.update_xaxes(title_text="Conversion rate", tickformat=".0%", row=1, col=2)
dashboard_fig.update_yaxes(title_text="Sales per lead ($)", row=1, col=2)
dashboard_fig.update_xaxes(title_text="Channel", row=2, col=1)
dashboard_fig.update_yaxes(title_text="Revenue lift ($)", row=2, col=1)

dashboard_fig.show()
dashboard_fig.write_html(str(OUTPUT_HTML), include_plotlyjs="cdn")
print(f"Dashboard exported to: {OUTPUT_HTML}")

Dashboard exported to: c:\Users\davie\OneDrive\Documentos\Data Analyst\marketing_campaign_dashboard_from_notebook.html


## Dashboard code explained

The dashboard HTML is organized in three layers:

1. **Head section**: loads the fonts, Plotly library, and all CSS styles used by the dashboard layout.
2. **Body section**: defines the page structure, including the hero recommendation block, KPI cards, chart containers, and the ranking table.
3. **Script section**: stores the dashboard data in JavaScript arrays and uses `Plotly.newPlot(...)` to render each chart.

How to read the code:

- The CSS variables in `:root` control the dashboard palette and visual style.
- The `comboData` array contains the ranked campaign and channel combinations.
- The `deltaData` array stores the Campaign B versus Campaign A uplift by channel.
- The `salesByCombo`, `efficiencyPlot`, and `toneDelta` charts are each created with Plotly using those arrays.
- The final table is built dynamically by looping through `comboData` and inserting rows into the table body.

The next cell loads the exact source code from `marketing_campaign_dashboard.html` and displays it directly in the notebook.

In [7]:
from IPython.display import Code, display

DASHBOARD_SOURCE_PATH = Path(r"c:\Users\davie\OneDrive\Documentos\Data Analyst\marketing_campaign_dashboard.html")
dashboard_html_code = DASHBOARD_SOURCE_PATH.read_text(encoding="utf-8")

print(f"Dashboard source loaded from: {DASHBOARD_SOURCE_PATH}")
display(Code(dashboard_html_code, language="html"))

Dashboard source loaded from: c:\Users\davie\OneDrive\Documentos\Data Analyst\marketing_campaign_dashboard.html


<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Marketing Campaign Dashboard</title>
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=Manrope:wght@400;500;600;700;800&family=Source+Serif+4:wght@600;700&display=swap" rel="stylesheet">
  <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
  <style>
    :root {
      --ink: #1d2a3a;
      --muted: #64748b;
      --paper: #f7f3eb;
      --card: rgba(255, 252, 247, 0.88);
      --line: rgba(29, 42, 58, 0.12);
      --navy: #20364f;
      --teal: #2f7f7b;
      --gold: #ca8f2c;
      --rose: #c96b59;
      --cream: #fffdf8;
      --shadow: 0 18px 50px rgba(32, 54, 79, 0.12);
    }

    * {
      box-sizing: border-box;
    }

    body {
      margin: 0;
      font-family: 'Manrope', sans-serif;
      color: var(--ink);
      background:
        radial-gradient(circle at top left, rgba(201, 107, 89, 0.18), transparent 28%),
        radial-gradient(circle at top right, rgba(47, 127, 123, 0.2), transparent 26%),
        linear-gradient(180deg, #f2ede4 0%, #f8f5ef 42%, #fbfaf7 100%);
    }

    .shell {
      max-width: 1400px;
      margin: 0 auto;
      padding: 32px 24px 56px;
    }

    .hero {
      position: relative;
      overflow: hidden;
      background: linear-gradient(135deg, rgba(255,255,255,0.92), rgba(247,243,235,0.78));
      border: 1px solid rgba(255,255,255,0.65);
      border-radius: 28px;
      box-shadow: var(--shadow);
      padding: 34px;
      display: grid;
      grid-template-columns: 1.35fr 0.9fr;
      gap: 28px;
    }

    .hero::after {
      content: "";
      position: absolute;
      inset: auto -80px -80px auto;
      width: 220px;
      height: 220px;
      border-radius: 999px;
      background: linear-gradient(135deg, rgba(202,143,44,0.25), rgba(47,127,123,0.12));
      filter: blur(8px);
    }

    .eyebrow {
      display: inline-flex;
      align-items: center;
      gap: 10px;
      padding: 8px 14px;
      border-radius: 999px;
      background: rgba(32, 54, 79, 0.08);
      color: var(--navy);
      font-size: 12px;
      font-weight: 800;
      letter-spacing: 0.12em;
      text-transform: uppercase;
    }

    h1 {
      margin: 16px 0 14px;
      font-family: 'Source Serif 4', serif;
      font-size: clamp(2.2rem, 3.6vw, 4.1rem);
      line-height: 0.96;
      letter-spacing: -0.04em;
    }

    .hero p {
      margin: 0;
      max-width: 760px;
      font-size: 1rem;
      line-height: 1.7;
      color: rgba(29, 42, 58, 0.84);
    }

    .recommendation-card {
      align-self: stretch;
      background: linear-gradient(160deg, rgba(32,54,79,0.95), rgba(47,127,123,0.92));
      color: white;
      border-radius: 24px;
      padding: 24px;
      display: flex;
      flex-direction: column;
      justify-content: space-between;
      box-shadow: 0 20px 40px rgba(32, 54, 79, 0.22);
    }

    .recommendation-card .label {
      font-size: 12px;
      text-transform: uppercase;
      letter-spacing: 0.14em;
      opacity: 0.78;
      font-weight: 700;
    }

    .recommendation-card h2 {
      margin: 10px 0 10px;
      font-family: 'Source Serif 4', serif;
      font-size: 2rem;
      line-height: 1.02;
      letter-spacing: -0.03em;
    }

    .recommendation-card p {
      color: rgba(255,255,255,0.84);
      line-height: 1.65;
      margin: 0 0 16px;
    }

    .impact-strip {
      display: grid;
      grid-template-columns: repeat(3, minmax(0, 1fr));
      gap: 10px;
    }

    .impact-pill {
      padding: 12px 14px;
      background: rgba(255,255,255,0.12);
      border: 1px solid rgba(255,255,255,0.16);
      border-radius: 18px;
    }

    .impact-pill strong {
      display: block;
      font-size: 1.1rem;
      margin-bottom: 4px;
    }

    .impact-pill span {
      font-size: 0.82re

## Dashboard metrics explained

The dashboard metrics are all designed to answer one business question:

**Which campaign and channel combination should we focus on to increase sales to new customers?**

### KPI cards

**1. New-Customer Sales: $988.5K**  
This is the total revenue generated by all new-customer interactions in the dataset. It represents the full sales opportunity being evaluated.

**2. Average Conversion: 47.86%**  
This is the baseline conversion rate across all new-customer traffic. It is the benchmark used to judge whether a campaign and channel combination is performing above or below average.

**3. B Email vs A Email: +$47.9K**  
This metric compares the sales generated by Campaign B Email versus Campaign A Email for new customers. It shows the incremental revenue advantage of the formal Campaign B tone in the Email channel.

**4. Tone Winner: Campaign B**  
This is a summary conclusion from the data. It means Campaign B, which uses the more formal and corporative tone, produces more total new-customer sales than Campaign A across every channel.

### Charts

**1. Revenue by campaign and channel**  
This chart compares total sales for Campaign A and Campaign B across Email, Instagram, and Web Banner. It identifies which combination produces the most revenue. In this case, Campaign B + Email is the strongest performer.

**2. Scale versus efficiency**  
This chart combines three performance signals:
- Conversion rate on the x-axis
- Sales per lead on the y-axis
- Lead volume through bubble size

It helps separate combinations that are efficient from combinations that are both efficient and large enough to matter commercially.

**3. Tone effect by channel**  
This chart compares Campaign B against Campaign A by channel using:
- Revenue lift in dollars
- Conversion delta in percentage points

It shows whether the more formal tone improves performance, and by how much, in each acquisition channel.

### Ranked table metrics

**1. Leads**  
The number of new-customer interactions in that campaign and channel combination.

**2. Conversion**  
The percentage of those leads that converted into a sale.

**3. Total Sales**  
The total revenue produced by that combination. This is the main business outcome metric.

**4. Sales / Lead**  
The average revenue generated per lead. This is an efficiency metric that combines conversion behavior and revenue impact.

**5. Tone**  
This labels the communication style used in the campaign:
- Campaign A: informal and conversational
- Campaign B: formal and corporative

### Why the dashboard recommends Campaign B + Email

Campaign B + Email is recommended because it performs well across the three most important dimensions:

- **Scale**: it reaches a large number of new-customer leads.
- **Conversion**: it converts above the overall new-customer average.
- **Revenue**: it generates the highest total sales of any campaign and channel combination.

### Important limitation

The dataset does not include media spend or acquisition cost. That means the dashboard supports a **revenue and conversion** decision, not a full **ROI or profitability** decision.